# 04 — Classificação via Rubrica com Llama**TCC: Detecção de Smishing em Idosos com Modelos de Linguagem Natural**Classificação por **rubrica**, no espírito de Dimario, Bacha e Butka (2024).> **Nota metodológica (GPT → Llama):** o projeto originalmente previa GPT> (OpenAI). Por restrições de custo e acessibilidade, optou-se por um modelo> aberto quantizado, alinhado ao objetivo de avaliar soluções gratuitas e> reprodutíveis: pesos públicos permitem replicação exata, o que uma API paga> não garante. **O resumo, o abstract e a seção 3.3 da monografia ainda citam> GPT e precisam ser atualizados.**### Como a rubrica funcionaEm vez de pedir uma probabilidade — que um modelo de 3B calibra mal — pede-seque o modelo marque **quais critérios de golpe estão presentes**. A pontuação écalculada de forma determinística a partir dos critérios marcados. Isso dáinterpretabilidade (dá para dizer ao usuário *por que* a mensagem é suspeita,o que alimenta as diretrizes da etapa 4.4.5), robustez e determinismo.Os sete critérios estão ancorados em Bortot et al. (2024, p. 13), citado naseção 3.2, e nos padrões descritos na seção 3.1 — ver `src/rubrica.py`.### Anti-vazamentoExemplos few-shot vêm **exclusivamente do treino**. A escolha do prompt é feitana **validação** — iterar o prompt olhando o resultado do teste seria vazamento.

## 1. Setup

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
# No Colab CADA notebook roda em um runtime próprio: instalar dependências
# em um notebook não vale para os outros. Por isso esta célula se repete em
# todos, e não existe um "notebook de instalação".

REPO = 'https://github.com/FelypeSR/TCC_Cristian.git'   # ← ajuste aqui

!git clone -q {REPO} /content/TCC_Cristian 2>/dev/null || (cd /content/TCC_Cristian && git pull -q)
!pip install -q -r /content/TCC_Cristian/requirements.txt

from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/TCC_Cristian/src')

import config as CFG
CFG.fixar_seeds()
CFG.criar_pastas()
CFG.resumo()

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import rubrica as R
import evaluation as ev

print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "INDISPONÍVEL"}')
if not torch.cuda.is_available():
    raise RuntimeError('Troque o runtime para GPU: Ambiente de execução → Alterar tipo.')

treino = pd.read_csv(CFG.SPLIT_FILES['train'], encoding='utf-8')
val    = pd.read_csv(CFG.SPLIT_FILES['val'],   encoding='utf-8')
teste  = pd.read_csv(CFG.SPLIT_FILES['test'],  encoding='utf-8')

print(f'\nTreino: {len(treino)} (fonte dos exemplos)  |  Val: {len(val)}  |  Teste: {len(teste)}')
display(R.tabela_criterios())

## 2. Carregamento do Llama quantizado em 4-bit

In [ ]:
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN, add_to_git_credential=False)

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

MODEL_ID = CFG.MODELOS['llama']
tok = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)

# padding à esquerda: obrigatório para geração em lote com modelos causais
tok.padding_side = 'left'
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

modelo = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map='auto', token=HF_TOKEN,
)
modelo.eval()

print(f'VRAM em uso: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

## 3. Exemplos few-shot — anotação manualOs exemplos precisam vir do **treino** e ter os critérios anotados à mão. Sãopoucos: seis mensagens. Anote com cuidado — eles definem o padrão que o modelovai imitar.Execute a célula abaixo para ver os candidatos sorteados do treino, depoispreencha `ANOTACOES` na célula seguinte.

In [ ]:
N_POR_CLASSE = 3

cand_pos = treino[treino[CFG.COL_ROTULO] == CFG.CLASSE_POSITIVA].sample(
    N_POR_CLASSE, random_state=CFG.SEED)
cand_neg = treino[treino[CFG.COL_ROTULO] == CFG.CLASSE_NEGATIVA].sample(
    N_POR_CLASSE, random_state=CFG.SEED)
candidatos = pd.concat([cand_pos, cand_neg]).reset_index(drop=True)

print('Anote os critérios de cada mensagem (ver tabela da seção 1):\n')
for i, linha in candidatos.iterrows():
    print(f'[{i}] ({linha[CFG.COL_ROTULO]}) {linha[CFG.COL_TEXTO]}')
    print()

In [ ]:
# ── Preencha: índice do candidato → lista de critérios presentes ───────────
# Registre os critérios REALMENTE presentes em cada mensagem exibida acima,
# no formato {índice: [ids dos critérios]}. Mensagem legítima costuma ficar
# com lista vazia. Deixar o dicionário vazio interrompe a execução logo abaixo.
#
# Modelo (apague e substitua pelos seus valores):
#     ANOTACOES = {0: [1, 2, 3], 1: [1, 3, 6], 2: [2, 4], 3: [], 4: [], 5: []}
ANOTACOES = {}

# ── Verificação da anotação ───────────────────────────────────────────────
# Estes exemplos são o few-shot: o que estiver aqui é o padrão que o modelo
# vai imitar em todo o conjunto de teste. Uma anotação inventada não quebra
# nada — ela apenas ensina a rubrica errada, e o efeito aparece diluído nas
# métricas finais, onde ninguém consegue mais atribuí-lo à causa. Por isso a
# execução para aqui, como a revisão manual do notebook 00 também para.
if not ANOTACOES:
    raise ValueError(
        'ANOTACOES está vazio — a anotação manual dos exemplos few-shot ainda '
        'não foi feita.\nLeia as mensagens impressas na célula acima e registre, '
        'para cada índice, os critérios presentes.'
    )

faltando = [i for i in candidatos.index if i not in ANOTACOES]
if faltando:
    raise ValueError(f'Índices sem anotação: {faltando}')

for indice, criterios in ANOTACOES.items():
    invalidos = [c for c in criterios if c not in R.CRITERIOS_POR_ID]
    if invalidos:
        raise ValueError(
            f'Índice {indice}: critérios inexistentes {invalidos}. '
            f'Válidos: 1 a {R.N_CRITERIOS}.'
        )

# Coerência entre a anotação e o rótulo — não bloqueia, mas avisa: uma
# mensagem de golpe sem nenhum critério, ou uma legítima com vários, costuma
# ser engano de digitação de índice.
for indice, criterios in ANOTACOES.items():
    rotulo = candidatos.loc[indice, CFG.COL_ROTULO]
    if rotulo == CFG.CLASSE_POSITIVA and not criterios:
        print(f'[AVISO] índice {indice} é {CFG.CLASSE_POSITIVA} e não tem critério algum')
    if rotulo == CFG.CLASSE_NEGATIVA and len(criterios) >= 3:
        print(f'[AVISO] índice {indice} é {CFG.CLASSE_NEGATIVA} e tem {len(criterios)} critérios')

exemplos = candidatos.copy()
exemplos['criterios'] = exemplos.index.map(ANOTACOES)
exemplos['resposta_esperada'] = exemplos['criterios'].apply(R.resposta_esperada)

# Intercala as classes para evitar viés de posição no prompt
exemplos = exemplos.sample(frac=1, random_state=CFG.SEED).reset_index(drop=True)

for _, e in exemplos.iterrows():
    print(f'  [{e["resposta_esperada"]:>12}] {e[CFG.COL_TEXTO][:80]}')

print('\n=== Prompt montado (final) ===')
print(tok.apply_chat_template(
    R.construir_mensagens('Mensagem de exemplo', exemplos),
    tokenize=False, add_generation_prompt=True)[-700:])

## 4. Inferência em loteProcessar uma mensagem por vez, com o limite de 90 min de ociosidade e 12 h desessão do Colab, pode simplesmente não terminar. O lote resolve isso, e ocheckpoint protege contra queda de sessão.

In [ ]:
from tqdm.auto import tqdm

BATCH = 8


def classificar_lote(textos, exemplos_fs):
    """Roda a rubrica em um lote de mensagens."""
    prompts = [
        tok.apply_chat_template(R.construir_mensagens(t, exemplos_fs),
                                tokenize=False, add_generation_prompt=True)
        for t in textos
    ]
    entrada = tok(prompts, return_tensors='pt', padding=True,
                  add_special_tokens=False).to(modelo.device)

    with torch.no_grad():
        saida = modelo.generate(
            **entrada,
            max_new_tokens=20,   # a resposta é uma lista curta de números
            do_sample=False,     # decodificação gulosa: determinística
            pad_token_id=tok.pad_token_id,
        )

    novos = saida[:, entrada['input_ids'].shape[-1]:]
    return [tok.decode(s, skip_special_tokens=True).strip() for s in novos]


def rodar(df, exemplos_fs, checkpoint=None, desc='Classificando'):
    """Aplica a rubrica a um DataFrame inteiro, com checkpoint opcional."""
    prontos = {}
    if checkpoint and os.path.isfile(checkpoint):
        anterior = pd.read_csv(checkpoint, encoding='utf-8')
        prontos = dict(zip(anterior['id'], anterior['resposta_bruta'].fillna('')))
        print(f'Checkpoint: {len(prontos)} mensagens já classificadas')

    pendentes = df[~df['id'].isin(prontos)].reset_index(drop=True)
    linhas = []

    for inicio in tqdm(range(0, len(pendentes), BATCH), desc=desc):
        pedaco = pendentes.iloc[inicio:inicio + BATCH]
        respostas = classificar_lote(pedaco[CFG.COL_TEXTO].tolist(), exemplos_fs)

        for (_, linha), bruta in zip(pedaco.iterrows(), respostas):
            prontos[linha['id']] = bruta

        if checkpoint and (inicio // BATCH) % 10 == 0:
            pd.DataFrame({'id': list(prontos), 'resposta_bruta': list(prontos.values())}) \
              .to_csv(checkpoint, index=False, encoding='utf-8')

    if checkpoint:
        pd.DataFrame({'id': list(prontos), 'resposta_bruta': list(prontos.values())}) \
          .to_csv(checkpoint, index=False, encoding='utf-8')

    for _, linha in df.iterrows():
        analise = R.classificar_resposta(prontos.get(linha['id'], ''))
        linhas.append({'id': linha['id'], **analise})

    return pd.DataFrame(linhas)


print('Funções de inferência definidas.')

## 5. Seleção do prompt na validaçãoCompara few-shot com zero-shot **na validação**. A versão escolhida aqui é aúnica que toca o conjunto de teste — é isso que impede o vazamento por iteraçãode prompt.

In [ ]:
variantes = {
    'few-shot (6 exemplos)': exemplos,
    'zero-shot':             None,
}

escolhido, melhor_f2, limiar_escolhido = None, -1.0, 0.5

for nome, exemplos_fs in variantes.items():
    resultado = rodar(val, exemplos_fs, desc=f'val · {nome}')
    juntos = val[['id', CFG.COL_ROTULO]].merge(resultado, on='id')

    limiar, f2 = ev.calibrar_limiar(juntos[CFG.COL_ROTULO], juntos['score'], beta=2)
    invalidas = int((~juntos['resposta_valida']).sum())

    print(f'\n{nome}')
    print(f'  F2 na validação  : {f2:.4f}  (limiar {limiar:.4f})')
    print(f'  respostas inválidas: {invalidas} / {len(juntos)} ({invalidas/len(juntos):.1%})')

    if f2 > melhor_f2:
        escolhido, melhor_f2, limiar_escolhido = exemplos_fs, f2, limiar

nome_escolhido = [k for k, v in variantes.items() if v is escolhido][0]
print(f'\n→ Prompt escolhido: {nome_escolhido}  (F2={melhor_f2:.4f}, limiar={limiar_escolhido:.4f})')

## 6. Inferência no conjunto de teste

In [ ]:
CHECKPOINT = f"{CFG.PATHS['predictions']}/llama_checkpoint.csv"

resultado = rodar(teste, escolhido, checkpoint=CHECKPOINT, desc='teste')
juntos = teste[['id', CFG.COL_ROTULO, 'tipo_golpe']].merge(resultado, on='id')

# Respostas fora do formato: contadas como abstenção e reportadas.
# Quando é preciso atribuir rótulo, atribui-se SMISHING — cair no rótulo
# negativo produziria justamente o erro mais caro do projeto.
invalidas = juntos[~juntos['resposta_valida']]
print(f'Respostas fora do formato: {len(invalidas)} / {len(juntos)} ({len(invalidas)/len(juntos):.1%})')
if len(invalidas):
    print('\nExemplos:')
    for _, linha in invalidas.head(5).iterrows():
        print(f'  id={linha["id"]}  bruta={linha["resposta_bruta"]!r}')

pred = ev.aplicar_limiar(juntos['score'], limiar_escolhido)
pred = np.where(juntos['resposta_valida'], pred, CFG.CLASSE_POSITIVA)

In [ ]:
y_real = juntos[CFG.COL_ROTULO].values
metricas = ev.calcular_metricas(y_real, pred, juntos['score'])

print('=== Llama (rubrica) — teste ===')
for chave in ev.ORDEM_METRICAS:
    valor = metricas[chave]
    print(f'  {ev.NOMES_METRICAS[chave]:16}: {valor:.4f}' if valor is not None
          else f'  {ev.NOMES_METRICAS[chave]:16}: –')
print(f"  VP={metricas['VP']}  FN={metricas['FN']}  FP={metricas['FP']}  VN={metricas['VN']}")

ev.salvar_predicoes(juntos['id'], y_real, pred, juntos['score'], 'llama')

ev.plot_confusao(y_real, pred,
                 f'Llama (rubrica)\nF2={metricas["f2"]:.3f}  Recall={metricas["recall"]:.3f}',
                 salvar_como='04_confusao_llama.png')
plt.show()

## 7. Níveis de risco e explicaçõesA seção 5 da monografia promete uma rubrica capaz de atribuir **níveis de risco**e servir como recurso educativo. Esta seção entrega as duas coisas, e é o insumodireto das diretrizes da etapa 4.4.5.

In [ ]:
juntos['nivel_risco'] = juntos['score'].apply(CFG.nivel_risco)

print('=== Distribuição dos níveis de risco no teste ===')
print(pd.crosstab(juntos['nivel_risco'], juntos[CFG.COL_ROTULO]).to_string())

print('\n=== Critérios mais acionados nas mensagens de golpe ===')
golpes = juntos[juntos[CFG.COL_ROTULO] == CFG.CLASSE_POSITIVA]
contagem = pd.Series(
    [nome for lista in golpes['criterios_nomes'] for nome in lista]).value_counts()
print(contagem.to_string())

juntos.to_csv(f"{CFG.PATHS['predictions']}/llama_rubrica_detalhado.csv",
              index=False, encoding='utf-8')

In [ ]:
# Exemplos de alerta como o usuário final o veria
print('=== Exemplos de alerta ao usuário (etapa 4.4.5) ===')
for _, linha in golpes.nlargest(3, 'score').iterrows():
    texto = teste.loc[teste['id'] == linha['id'], CFG.COL_TEXTO].iloc[0]
    print(f'\nMensagem: {texto}')
    print(f'Risco: {linha["nivel_risco"].upper()} (pontuação {linha["score"]:.2f})')
    print(R.explicar(linha['criterios'], linha['nivel_risco']))
    print('-' * 70)

print('\nProssiga para o notebook 05_avaliacao.ipynb.')